<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/1C_basic_planning_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1C — Planificación reactiva (ReAct) y jerárquica (Manager–Workers)
**ISY0101 · Ingeniería de Soluciones con IA · 2.3.1 Estrategias de planificación y orquestación**

**Dos arquitecturas, el mismo caso y las mismas fuentes.** En la primera, el LLM escoge una herramienta **tras observar** cada resultado. En la segunda, un **manager** descompone un objetivo y delega análisis a **dos agentes especializados**. Ambas se pueden comparar con 1B (*Plan-and-Execute*).

**Al terminar:** (1) distinguir decisiones reactivas de un plan anticipado; (2) identificar responsabilidades del manager y de los workers; (3) diferenciar planificación jerárquica y orquestación jerárquica; (4) discutir trazabilidad, costos y riesgos.

> **Importante:** los datos del caso son **ficticios y preparados con fines didácticos**. No hay dependencia de Wikipedia/Wikidata, que causó fallos en versiones anteriores. Las llamadas al LLM sí requieren acceso a Mistral. Este ejercicio implementa un ciclo *tipo ReAct* con acciones y observaciones visibles; no expone razonamientos internos del modelo.

## 1. Setup: misma convención que los notebooks anteriores
En Google Colab, define **`LLM_API_KEY`** en el panel 🔑 *Secrets* y concede permiso al notebook. En ejecución local usa `.env`. Opcional: `LLM_BASE_URL` y `LLM_MODEL`. No se muestran las claves.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip install -q langchain-openai python-dotenv
# Local: pip install langchain-openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.3/573.3 kB 14.4 MB/s eta 0:00:00


In [2]:
import os
try:
    from google.colab import userdata
    for key in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            value = userdata.get(key)
            if value:
                os.environ[key] = value
        except Exception:
            pass
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
if not os.getenv("LLM_API_KEY"):
    raise RuntimeError("Configura LLM_API_KEY en Colab Secrets o en .env")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model=os.environ["LLM_MODEL"], base_url=os.environ["LLM_BASE_URL"],
                 api_key=os.environ["LLM_API_KEY"], temperature=0, timeout=45, max_retries=2)
print("Modelo configurado:", os.environ["LLM_MODEL"], "| clave protegida")

Modelo configurado: ministral-14b-latest | clave protegida


## 2. Caso común y herramientas verificables
**Caso ficticio:** «MercadoAndes» es un comercio digital que desea mejorar ventas y servicio. El objetivo es **diagnosticar causas y recomendar dos acciones priorizadas**, con cifras comprobables. Las fuentes son registros preparados en Python (no páginas externas). Así, los errores del LLM no se confunden con caídas de APIs.

Disponibles: `ver_ventas`, `ver_reclamos` y `ver_capacidad`. Cada una retorna **datos concretos** y etiquetas de fuente. No existe una herramienta que entregue automáticamente las recomendaciones.

In [3]:
import json, re, copy
from pprint import pprint

OBJETIVO = ("Diagnosticar por qué MercadoAndes tuvo problemas durante el último trimestre "
            "y recomendar dos medidas priorizadas basadas en evidencia de ventas, reclamos y capacidad operativa.")

# Datos inventados para clase, NO datos empresariales reales.
DATOS = {
    "ver_ventas": {"fuente": "VENTAS_SIMULADAS", "trimestre_previo": 1200, "trimestre_actual": 1020,
                   "carritos_abandonados_pct": 32, "nota": "Pedidos finalizados por trimestre"},
    "ver_reclamos": {"fuente": "RECLAMOS_SIMULADOS", "reclamos_totales": 230,
                     "reclamos_entrega_tardia": 138, "reclamos_producto": 46, "otros": 46},
    "ver_capacidad": {"fuente": "OPERACION_SIMULADA", "despachos_semanales": 400,
                      "capacidad_semanal": 320, "entregas_tardias_pct": 27}
}
DESCRIPCIONES = {
    "ver_ventas": "Volumen trimestral y abandono de carritos",
    "ver_reclamos": "Reclamos por causa",
    "ver_capacidad": "Demanda, capacidad logística y retrasos"
}

def usar_herramienta(nombre):
    if nombre not in DATOS:
        raise ValueError(f"Herramienta no autorizada: {nombre}")
    return copy.deepcopy(DATOS[nombre])

print("Objetivo:", OBJETIVO)
print("Herramientas:", DESCRIPCIONES)

Objetivo: Diagnosticar por qué MercadoAndes tuvo problemas durante el último trimestre y recomendar dos medidas priorizadas basadas en evidencia de ventas, reclamos y capacidad operativa.
Herramientas: {'ver_ventas': 'Volumen trimestral y abandono de carritos', 'ver_reclamos': 'Reclamos por causa', 'ver_capacidad': 'Demanda, capacidad logística y retrasos'}


## 3. Utilidad para respuestas estructuradas
Solicitamos **JSON**, verificamos los campos y damos una oportunidad de corregir errores de formato. **No exigimos un plan fijo**. El código valida estructura y permisos; el juicio de calidad de las recomendaciones sigue siendo responsabilidad de quien revisa el ejercicio.

In [7]:
def extraer_json(texto):
    if not isinstance(texto, str):
        raise ValueError("La respuesta no es texto")
    limpio = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", texto.strip(), flags=re.I)
    try:
        value = json.loads(limpio, strict=False)
    except json.JSONDecodeError:
        inicio, fin = limpio.find("{"), limpio.rfind("}")
        if inicio < 0 or fin < inicio:
            raise ValueError("No se encontró objeto JSON")
        value = json.loads(limpio[inicio:fin+1], strict=False)
    if not isinstance(value, dict):
        raise ValueError("Se esperaba un objeto JSON")
    return value

def pedir_json(mensajes, validar, intentos=2):
    historial = list(mensajes)
    for intento in range(intentos):
        respuesta = llm.invoke(historial).content
        try:
            obj = extraer_json(respuesta)
            validar(obj)
            return obj
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as exc:
            if intento == intentos - 1:
                raise RuntimeError(f"El LLM no entregó JSON válido tras {intentos} intentos: {exc}") from exc
            historial.extend([("assistant", respuesta),
                              ("human", f"Corrige la salida. Error: {exc}. Responde SOLO un objeto JSON válido.")])

def exigir(condicion, mensaje):
    if not condicion:
        raise ValueError(mensaje)

## 4. Experimento A — Planificación **reactiva** (ciclo tipo ReAct)
Aquí **no pedimos un plan completo al inicio**. En cada iteración el agente lee las observaciones acumuladas y decide **una sola acción**: consultar una herramienta aún no usada o finalizar.

**Observación → decisión de acción → herramienta → nueva observación.** Cuando decide finalizar, una llamada separada al LLM redacta el informe utilizando la evidencia reunida. Esta separación evita exigir al mismo JSON una decisión de control y un informe extenso. Hay un máximo de seis ciclos y no se repiten herramientas.


In [5]:
HERRAMIENTAS = list(DATOS)

def validar_decision(obj):
    exigir(obj.get("accion") in ("consultar", "finalizar"), "accion debe ser consultar/finalizar")
    exigir(isinstance(obj.get("motivo_breve"), str), "Falta motivo_breve")
    if obj["accion"] == "consultar":
        exigir(obj.get("herramienta") in HERRAMIENTAS, "herramienta no permitida")
    else:
        # "finalizar" es una decisión de control; el informe se redacta aparte.
        # Así un campo opcional ausente no invalida una decisión correcta.
        pass

def ejecutar_reactivo(max_ciclos=6):
    observaciones = {}
    traza = []
    for ciclo in range(1, max_ciclos + 1):
        entrada = {
            "objetivo": OBJETIVO, "herramientas": DESCRIPCIONES,
            "observaciones": observaciones,
            "herramientas_ya_usadas": list(observaciones), "ciclos_restantes": max_ciclos-ciclo+1
        }
        mensajes = [
            ("system", "Eres un agente reactivo de diagnóstico. Decide SOLO la siguiente acción, "
             "tras ver las observaciones. No prepares una lista completa de pasos. "
             "Consulta fuentes suficientes antes de terminar (ventas, reclamos y capacidad). "
             "Nunca inventes cifras; cita el nombre de cada fuente. "
             "Devuelve SOLO JSON: si consultas, "
             "{\"accion\":\"consultar\",\"herramienta\":\"ver_ventas\",\"motivo_breve\":\"...\"}; "
             "si finalizas, "
             "{\"accion\":\"finalizar\",\"motivo_breve\":\"...\"}. "
             "Cuando ya estén las tres fuentes, elige finalizar. "
             "NO redactes el informe en la decisión; se generará en una llamada separada."),
            ("human", json.dumps(entrada, ensure_ascii=False))
        ]
        decision = pedir_json(mensajes, validar_decision)
        print(f"\nCiclo {ciclo} | Acción: {decision['accion']} | Motivo: {decision['motivo_breve']}")
        if decision["accion"] == "finalizar":
            if set(observaciones) != set(HERRAMIENTAS):
                print("Control: aún faltan fuentes; se continúa la exploración.")
                traza.append({"ciclo": ciclo, "accion": "finalizar_prematuro"})
                continue
            # Separar la redacción de la decisión permite comprobar cada etapa.
            informe = llm.invoke([
                ("system", "Redacta un informe ejecutivo breve para MercadoAndes con diagnóstico, "
                 "exactamente DOS acciones priorizadas y un indicador para cada una. "
                 "Utiliza EXCLUSIVAMENTE la evidencia proporcionada, cita las etiquetas de fuente, "
                 "no inventes cifras y distingue correlaciones de causas demostradas."),
                ("human", json.dumps({"objetivo": OBJETIVO, "evidencia": observaciones}, ensure_ascii=False))
            ]).content
            if not isinstance(informe, str) or not informe.strip():
                raise RuntimeError("El LLM no entregó texto para el informe final")
            print("\nINFORME REACTIVO:\n", informe)
            traza.append({"ciclo": ciclo, "accion": "finalizar", "informe_generado": True})
            return {"informe": informe, "observaciones": observaciones, "traza": traza}
        herramienta = decision["herramienta"]
        if herramienta in observaciones:
            print("Control: consulta repetida; se omite y se solicita otra decisión.")
            traza.append({"ciclo": ciclo, "accion": "repetición_rechazada", "herramienta": herramienta})
            continue
        resultado = usar_herramienta(herramienta)
        observaciones[herramienta] = resultado
        traza.append({"ciclo": ciclo, "accion": "consultar", "herramienta": herramienta, "observacion": resultado})
        print("Observación:", resultado)
    print("\nLímite de ciclos alcanzado: no hubo informe final.")
    return {"informe": None, "observaciones": observaciones, "traza": traza}

resultado_reactivo = ejecutar_reactivo()


Ciclo 1 | Acción: consultar | Motivo: Analizar tendencias de volumen trimestral y tasa de abandono de carritos para identificar posibles caídas en demanda o problemas en el embudo de conversión.
Observación: {'fuente': 'VENTAS_SIMULADAS', 'trimestre_previo': 1200, 'trimestre_actual': 1020, 'carritos_abandonados_pct': 32, 'nota': 'Pedidos finalizados por trimestre'}

Ciclo 2 | Acción: consultar | Motivo: Analizar causas de insatisfacción del cliente que puedan explicar la caída en ventas (1200 → 1020) y el alto abandono de carritos (32%).
Observación: {'fuente': 'RECLAMOS_SIMULADOS', 'reclamos_totales': 230, 'reclamos_entrega_tardia': 138, 'reclamos_producto': 46, 'otros': 46}

Ciclo 3 | Acción: consultar | Motivo: Evaluar correlación entre caída en ventas (1020 vs 1200), alta tasa de abandono de carritos (32%) y reclamos por entrega tardía (138/230) con capacidad logística, demanda real y retrasos operativos para identificar cuellos de botella críticos.
Observación: {'fuente': 'OPERAC

### ¿Qué observar?
- El LLM **no ve** el resultado de una herramienta hasta ejecutarla.
- Puede modificar su decisión siguiente tras observar nuevos datos.
- La decisión `finalizar` se valida de manera independiente del informe; la redacción ocurre después.
- La política de seguridad evita repetir consultas, controla herramientas y exige evidencia mínima antes de finalizar.
- Este agente reactivo sigue siendo **un agente que usa herramientas**, no un equipo multiagente.


## 5. Experimento B — Planificación **jerárquica** + orquestación jerárquica
Para el **mismo objetivo**, creamos tres roles con llamadas separadas al LLM:

- **Manager:** descompone el objetivo y delega dos tareas especializadas (planificación jerárquica).
- **Agente comercial:** analiza ventas y reclamos.
- **Agente operativo:** analiza capacidad y reclamos.

El manager **recibe informes** y redacta el diagnóstico integrado (orquestación jerárquica). El orden de las dos delegaciones no es esencial; en este ejemplo se ejecutan secuencialmente para mostrar su traza con facilidad. Los workers reciben **solo fuentes autorizadas** y no utilizan datos inventados. Cada rol es una instancia lógica diferenciada por sus instrucciones y su invocación; comparten el mismo modelo base.

In [8]:
ROLES = {
    "comercial": {"mision": "Analizar comportamiento comercial y fricciones del cliente",
                  "fuentes": ["ver_ventas", "ver_reclamos"]},
    "operaciones": {"mision": "Analizar cuello de botella logístico y medidas operativas",
                   "fuentes": ["ver_capacidad", "ver_reclamos"]},
}

def validar_delegacion(obj):
    tareas = obj.get("delegaciones")
    exigir(isinstance(tareas, list) and len(tareas) == 2, "Se exigen dos delegaciones")
    roles = [t.get("rol") for t in tareas if isinstance(t, dict)]
    exigir(set(roles) == set(ROLES) and len(roles) == 2, "Un encargo por rol")
    exigir(all(isinstance(t.get("objetivo"), str) and t["objetivo"].strip() for t in tareas),
           "Cada delegación necesita objetivo")

manager_instruccion = ("Actúas como Manager de un equipo de dos agentes analistas. "
    "Divide el objetivo en dos encargos complementarios para los roles 'comercial' y 'operaciones'. "
    "No redactes el informe final todavía. Devuelve SOLO JSON: "
    "{\"delegaciones\":[{\"rol\":\"comercial\",\"objetivo\":\"...\"},"
    "{\"rol\":\"operaciones\",\"objetivo\":\"...\"}]}")

plan_manager = pedir_json([
    ("system", manager_instruccion),
    ("human", f"Objetivo general: {OBJETIVO}\nRoles: {json.dumps(ROLES, ensure_ascii=False)}")
], validar_delegacion)
print("PLAN DE DELEGACIÓN DEL MANAGER:")
pprint(plan_manager)

PLAN DE DELEGACIÓN DEL MANAGER:
{'delegaciones': [{'objetivo': '1. **Analizar el desempeño comercial y su '
                               'impacto en ventas:**\n'
                               '      - Evaluar la evolución trimestral de '
                               '**ventas totales, ticket promedio y margen por '
                               'línea de producto/región** utilizando '
                               '*ver_ventas*.\n'
                               '      - Identificar **segmentos de clientes '
                               'con mayor caída** (B2B/B2C, tamaño, frecuencia '
                               'de compra) y correlacionar con **tasa de '
                               'conversión y abandono de carrito**.\n'
                               '      - **Cruzar con reclamos** '
                               '(*ver_reclamos*): Determinar si problemas '
                               'comerciales (ej: promociones no comunicadas, '
                               

In [9]:
def ejecutar_worker(rol, tarea):
    definicion = ROLES[rol]
    evidencia = {herr: usar_herramienta(herr) for herr in definicion["fuentes"]}
    instrucciones = (
        f"Eres un agente especializado en {rol}. Tu misión: {definicion['mision']}. "
        "Analiza solamente la evidencia suministrada. No inventes datos ni atribuyas causalidad "
        "con certeza cuando solo existan indicios. Entrega: hallazgos con cifras, "
        "una recomendación y una incertidumbre. Cita las etiquetas 'fuente'."
    )
    mensaje = f"Encargo del manager: {tarea}\nEvidencia: {json.dumps(evidencia, ensure_ascii=False)}"
    respuesta = llm.invoke([("system", instrucciones), ("human", mensaje)]).content
    return {"rol": rol, "encargo": tarea, "evidencia": evidencia, "informe": respuesta}

informes_workers = []
for delegacion in plan_manager["delegaciones"]:
    rol = delegacion["rol"]
    print(f"\nMANAGER → {rol}: {delegacion['objetivo']}")
    informe = ejecutar_worker(rol, delegacion["objetivo"])
    informes_workers.append(informe)
    print(f"AGENTE {rol.upper()} → MANAGER:\n{informe['informe'][:1800]}\n")


MANAGER → comercial: 1. **Analizar el desempeño comercial y su impacto en ventas:**
      - Evaluar la evolución trimestral de **ventas totales, ticket promedio y margen por línea de producto/región** utilizando *ver_ventas*.
      - Identificar **segmentos de clientes con mayor caída** (B2B/B2C, tamaño, frecuencia de compra) y correlacionar con **tasa de conversión y abandono de carrito**.
      - **Cruzar con reclamos** (*ver_reclamos*): Determinar si problemas comerciales (ej: promociones no comunicadas, stock no disponible) generaron quejas recurrentes. Priorizar categorías/productos con **mayor impacto en ventas y reclamos simultáneos**.

      2. **Detectar fricciones en el funnel de compra:**
      - Analizar etapas con **mayor tasa de drop-off** (ej: checkout, post-compra) usando métricas de *ver_ventas* (ej: errores en pago, tiempos de carga).
      - Extraer **motivos recurrentes de reclamos** (*ver_reclamos*) relacionados con la experiencia de compra (ej: 'entrega tardía', 

In [10]:
sintesis_manager = llm.invoke([
    ("system", "Eres el manager responsable de integrar análisis de agentes. "
     "Escribe un informe final corto con diagnóstico, DOS acciones priorizadas, "
     "indicadores para medir resultados y advertencias. "
     "Usa solo la evidencia disponible, cita las fuentes, evita inventar cifras "
     "y distingue indicios de causalidad comprobada."),
    ("human", "Objetivo: " + OBJETIVO + "\nInformes y evidencias:\n" +
     json.dumps(informes_workers, ensure_ascii=False))
]).content
print("INFORME INTEGRADO DEL MANAGER:\n")
print(sintesis_manager)

INFORME INTEGRADO DEL MANAGER:

---
**Informe Final: Diagnóstico y Acciones Priorizadas para MercadoAndes**
**Fecha:** [Incluir fecha]
**Responsable:** [Tu nombre/rol]

---

### **1. Diagnóstico (Basado en Evidencia Disponible)**
**Problema central:**
Caída del **15% en ventas trimestrales** (de 1200 a 1020 unidades) acompañada de:
- **32% de abandono de carrito** (*ver_ventas*).
- **230 reclamos totales**, con **138 (60%) por entregas tardías** y **46 (20%) por productos no disponibles** (*ver_reclamos*).
- **Sobrecapacidad operativa**: Demanda de **400 despachos/semana** vs. capacidad de **320** (*ver_capacidad*), resultando en **27% de entregas tardías**.

**Indicios de causalidad comprobada (correlación directa):**
- **Entregas tardías (138 reclamos)** → Vinculadas al **27% de entregas tardías** por sobrecapacidad logística (*ver_capacidad*).
- **Productos no disponibles (46 reclamos)** → Posiblemente relacionado con el **32% de abandono de carrito** (clientes que no encuentran sto

## 6. Comparación y discusión
| Aspecto | Reactivo (A) | Jerárquico (B) |
|---|---|---|
| ¿Cuándo se decide qué hacer? | En cada ciclo, tras observar | El manager asigna encargos antes del trabajo de los especialistas |
| ¿Quién analiza? | Un agente con herramientas | Workers especializados con instrucciones propias |
| ¿Cómo se coordinan? | Estado y ciclo del mismo agente | Manager delega y luego integra informes |
| ¿Hay planificación global explícita? | No necesariamente | Sí, mediante delegaciones de alto nivel |
| ¿Hay múltiples agentes LLM? | No | Sí, roles lógicos de manager y workers |

**Advertencia conceptual:** planificación jerárquica = **descomponer objetivos**. Orquestación jerárquica = **coordinar quién realiza cada subobjetivo y reunir resultados**. En el experimento B ocurren ambas.

### Preguntas para los estudiantes
1. ¿Qué observación concreta condicionó la siguiente acción del experimento reactivo?
2. ¿Qué ventaja y qué costo tiene solicitar varias respuestas al LLM?
3. ¿Podrían los dos workers trabajar en paralelo? ¿Qué problema surgiría si tuvieran que modificar el mismo recurso?
4. ¿Por qué no podemos concluir causalidad solo a partir de la coincidencia entre saturación y entregas tardías?
5. Si falla una herramienta en el experimento A, ¿qué información necesitaría el sistema para recuperarse?

**Nota:** los modelos pueden variar sus respuestas entre ejecuciones. Las validaciones controlan esquema y accesos, no garantizan corrección semántica de los informes. No se ha realizado una ejecución en vivo con Mistral en este entorno.